# Sesión 2 (15 sep 2026): Pandas y procesamiento de datos

Objetivos: cargar, limpiar, validar y agregar datos con **pandas** y `pathlib`.


In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA = Path("Datos") / "ventas.csv"
df = pd.read_csv(DATA)
df.head(), df.dtypes, df.shape


## Limpieza y validación


In [ ]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Separa filas válidas e inválidas según reglas de negocio."""
    work = frame.copy()
    work["unidades"] = pd.to_numeric(work["unidades"], errors="coerce")
    work["precio_unitario"] = pd.to_numeric(work["precio_unitario"], errors="coerce")
    mask_ok = work["unidades"].notna() & (work["unidades"] > 0) & (work["precio_unitario"] > 0)
    validos = work.loc[mask_ok].copy()
    errores = work.loc[~mask_ok].copy()
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores

validos, errores = validar_ventas(df)
print("válidos:", len(validos), "errores:", len(errores))
errores


## Agregaciones


In [ ]:
por_region = validos.groupby("region", as_index=False)["importe"].sum().sort_values("importe", ascending=False)
top_productos = (
    validos.groupby("producto", as_index=False)["importe"].sum()
    .sort_values("importe", ascending=False).head(3)
)
clientes_recurrentes = (
    validos.groupby("cliente_id").size().reset_index(name="compras")
    .query("compras > 1")
)
por_region, top_productos, clientes_recurrentes


## Exportación e informe de calidad


In [ ]:
out_csv = Path("Datos") / "ventas_limpias.csv"
out_json = Path("Datos") / "calidad_datos.json"
validos.to_csv(out_csv, index=False)
informe = {
    "filas_totales": int(len(df)),
    "filas_validas": int(len(validos)),
    "filas_invalidas": int(len(errores)),
    "importe_total": float(validos["importe"].sum()),
}
out_json.write_text(json.dumps(informe, indent=2, ensure_ascii=False), encoding="utf-8")
informe
